# 👤 myHR Assistant — Archery Canada Employee Portal
A RAG-based HR chatbot using FAISS, LiteLLM, and Gradio.

## Imports & Environment

In [ ]:
import os
import io
import requests
import numpy as np
import faiss
import gradio as gr
from pypdf import PdfReader
from litellm import embedding, completion
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()
print('Imports successful')

## Ingest: Scrape PDF → Markdown

In [ ]:
PDF_URL = "https://archerycanada.ca/wp-content/uploads/2019/07/Human-Resource-Policy-Manual.pdf"
MD_OUTPUT_PATH = "HR_Manual.md"


class ManualLoader:
    """
    Downloads a PDF from a URL, extracts text page-by-page,
    saves it as a Markdown file, and provides chunking.
    """

    def __init__(self, url: str = PDF_URL, md_path: str = MD_OUTPUT_PATH):
        self.url = url
        self.md_path = md_path

    def _download_pdf_bytes(self) -> bytes:
        print(f"Downloading PDF from:\n  {self.url}")
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(self.url, headers=headers, timeout=60)
        resp.raise_for_status()
        print(f"   ✔ Downloaded {len(resp.content):,} bytes")
        return resp.content

    def _pdf_bytes_to_markdown(self, pdf_bytes: bytes) -> str:
        reader = PdfReader(io.BytesIO(pdf_bytes))
        md_lines = []
        for i, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            md_lines.append(f"## Page {i}\n")
            md_lines.append(text.strip())
            md_lines.append("\n")
        return "\n".join(md_lines)

    def document_load(self) -> str:
        """Downloads the PDF, converts to Markdown, saves it, and returns the text."""
        if os.path.exists(self.md_path):
            print(f"Loading cached Markdown from '{self.md_path}'")
            with open(self.md_path, "r", encoding="utf-8") as f:
                return f.read()

        pdf_bytes = self._download_pdf_bytes()
        markdown_text = self._pdf_bytes_to_markdown(pdf_bytes)

        with open(self.md_path, "w", encoding="utf-8") as f:
            f.write(markdown_text)
        print(f"Saved Markdown to '{self.md_path}' ({len(markdown_text):,} chars)")
        return markdown_text

    def create_chunks(self, text: str, chunk_size: int = 1500, overlap: int = 500) -> list:
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunks.append(text[start:end])
            start += chunk_size - overlap
        return chunks


print('ManualLoader defined')

## Retrieve: FAISS Vector Store

In [ ]:
class ManualRetriever:
    """
    Embeds text chunks with OpenAI text-embedding-3-small via LiteLLM
    and stores them in a FAISS index for fast similarity search.
    """

    def __init__(self):
        self.index = None
        self.chunks = []

    def create_retriever(self, text_chunks: list):
        self.chunks = text_chunks
        print(f"🔢 Embedding {len(text_chunks)} chunks…")
        response = embedding(model="text-embedding-3-small", input=text_chunks)
        embeddings = np.array(
            [r["embedding"] for r in response["data"]], dtype="float32"
        )
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings)
        print(f"   ✔ FAISS index built (dim={dimension}, vectors={self.index.ntotal})")
        return self

    def get_relevant_chunks(self, query: str, k: int = 4) -> list:
        query_enc = embedding(model="text-embedding-3-small", input=[query])
        query_vec = np.array(
            [query_enc["data"][0]["embedding"]], dtype="float32"
        )
        distances, indices = self.index.search(query_vec, k)
        return [self.chunks[i] for i in indices[0] if i != -1]


print('ManualRetriever defined')

## Generate: RAG Response with Conversation History

In [ ]:
class GenerateResponse:
    """
    Contextualises the user query against conversation history,
    retrieves relevant chunks, and generates an answer via GPT-4o-mini.
    """

    def __init__(self, retriever: ManualRetriever):
        self.retriever = retriever

    def generate_relevant_text(self, query: str, history: list = None) -> str:
        """
        history: list of LiteLLM/OpenAI format dicts
                 [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
        """
        if history is None:
            history = []
        if len(query) < 10:
            raise ValueError("Query too short (min 10 characters).")

        # 1. Contextualise: rephrase query as standalone search query
        search_query = query
        if history:
            history_summary = "\n".join(
                [f"{m['role']}: {m['content']}" for m in history[-3:]]
            )
            rephrase_messages = [
                {
                    "role": "system",
                    "content": (
                        "Given the conversation history and a new question, "
                        "rephrase the question to be a standalone search query. "
                        "Do not answer it, just rephrase it."
                    ),
                },
                {
                    "role": "user",
                    "content": f"History:\n{history_summary}\n\nQuestion: {query}",
                },
            ]
            rephrase_res = completion(model="gpt-4o-mini", messages=rephrase_messages)
            search_query = rephrase_res.choices[0].message.content

        # 2. Retrieve
        context_chunks = self.retriever.get_relevant_chunks(search_query)
        context_text = "\n\n".join(context_chunks)

        # 3. Build messages
        messages = [
            {
                "role": "system",
                "content": (
                    "You are 'myHR', a knowledgeable, friendly HR assistant representing the company Archery Canada. "
                    "You are chatting with a user about Archery Canada."
                    "If relevant, use the given context to answer any question."
                    "Answer using ONLY the context provided. If the context doesn't contain "
                    "the answer, direct the employee to contact the HR Department. Max 3 sentences."
                ),
            }
        ]
        messages.extend(history)
        messages.append(
            {
                "role": "user",
                "content": f"Context from HR Manual:\n{context_text}\n\nQuestion: {query}",
            }
        )

        # 4. Generate
        response = completion(model="gpt-4o-mini", messages=messages)
        return response.choices[0].message.content


print('GenerateResponse defined')

## Build the RAG Engine

In [ ]:
loader = ManualLoader()
full_text = loader.document_load()
chunks = loader.create_chunks(full_text)
print(f"Total chunks: {len(chunks)}")

retriever = ManualRetriever().create_retriever(chunks)
rag_engine = GenerateResponse(retriever)
print('\nRAG engine ready!')

## Gradio Chat UI

In [ ]:
def chat_fn(message: str, gradio_history: list):
    """
    Gradio passes history as a list of [user_msg, assistant_msg] pairs.
    We convert it to the OpenAI/LiteLLM message format for the RAG engine.
    """
    litellm_history = []
    for user_msg, assistant_msg in gradio_history:
        if user_msg:
            litellm_history.append({"role": "user", "content": user_msg})
        if assistant_msg:
            litellm_history.append({"role": "assistant", "content": assistant_msg})

    try:
        answer = rag_engine.generate_relevant_text(message, history=litellm_history)
    except ValueError as e:
        answer = f"⚠️ {e}"
    except Exception as e:
        answer = f"❌ An error occurred: {e}"

    return answer


with gr.Blocks(title="myHR Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 👤 myHR: Archery Canada Employee Portal
        Hi there, I'm here to help answer any question about **Our HR Policy**.
        """
    )
    chatbot = gr.ChatInterface(
        fn=chat_fn,
        chatbot=gr.Chatbot(height=480, label="myHR Assistant"),
        textbox=gr.Textbox(
            placeholder="e.g. What is the vacation policy?",
            label="Your question",
            lines=1,
        ),
        examples=[
            "What is the vacation leave policy?",
            "How do I report a workplace harassment incident?",
            "What are the working hours?",
        ],
        submit_btn="Ask myHR",
    )

demo.launch(share=False)

---
# Evaluation Suite

## Eval Imports & Pydantic Models

In [ ]:
import json
import math
from pydantic import BaseModel, Field

EVAL_MODEL = "gpt-4.1-nano"   # LLM-as-a-judge model
TESTS_FILE = "tests.jsonl"    # Path to your JSONL test file


# Data model for a single test case  (mirrors test.py → TestQuestion)
class TestQuestion(BaseModel):
    """A test question with expected keywords and reference answer."""
    question:         str        = Field(description="The question to ask the RAG system")
    keywords:         list[str]  = Field(description="Keywords that must appear in retrieved context")
    reference_answer: str        = Field(description="The reference answer for this question")
    category:         str        = Field(description="Question category (e.g., direct_fact, spanning, temporal)")



# Structured output models for eval results  
class RetrievalEval(BaseModel):
    """Evaluation metrics for retrieval performance."""
    mrr:              float = Field(description="Mean Reciprocal Rank — average across all keywords")
    ndcg:             float = Field(description="Normalized Discounted Cumulative Gain (binary relevance)")
    keywords_found:   int   = Field(description="Number of keywords found in top-k results")
    total_keywords:   int   = Field(description="Total number of keywords to find")
    keyword_coverage: float = Field(description="Percentage of keywords found")


class AnswerEval(BaseModel):
    """LLM-as-a-judge evaluation of answer quality."""
    feedback:     str   = Field(description="Concise feedback comparing generated answer to reference")
    accuracy:     float = Field(description="Factual correctness vs reference answer (1 = wrong, 5 = perfect)")
    completeness: float = Field(description="Coverage of all reference answer aspects (1–5)")
    relevance:    float = Field(description="How directly the answer addresses the question (1–5)")


print('Eval models defined')

## Helpers: load_tests, fetch_context, answer_question

In [ ]:
# load_tests — reads JSONL file produced alongside this notebook
def load_tests(path: str = TESTS_FILE) -> list[TestQuestion]:
    """Load test questions from a JSONL file."""
    tests = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                tests.append(TestQuestion(**json.loads(line)))
    print(f"📋 Loaded {len(tests)} test(s) from '{path}'")
    return tests


# fetch_context — thin wrapper around ManualRetriever
# Returns plain string chunks (k=10).
# The retriever stores chunks as plain strings, so no .page_content needed.
def fetch_context(query: str, k: int = 10) -> list[str]:
    """Retrieve the top-k context chunks for a query."""
    return retriever.get_relevant_chunks(query, k=k)


# answer_question — runs the full RAG pipeline and also returns the chunks
def answer_question(query: str) -> tuple[str, list[str]]:
    """Run the RAG pipeline. Returns (generated_answer, retrieved_chunks)."""
    retrieved_chunks = fetch_context(query, k=10)
    # Build a minimal one-shot call (no history) just like the chat UI does
    context_text = "\n\n".join(retrieved_chunks)
    messages = [
        {
            "role": "system",
            "content": (
                "You are 'myHR', the official HR Assistant for Archery Canada. "
                "Answer using ONLY the context provided. If the context doesn't contain "
                "the answer, direct the employee to contact the HR Department. Max 3 sentences."
            ),
        },
        {
            "role": "user",
            "content": f"Context from HR Manual:\n{context_text}\n\nQuestion: {query}",
        },
    ]
    response = completion(model="gpt-4o-mini", messages=messages)
    answer = response.choices[0].message.content
    return answer, retrieved_chunks


print('Helper functions defined')

## Retrieval & Answer Evaluation Functions

In [ ]:
# Low-level metric helpers
# Chunks are plain strings — keyword search done directly on the string.
def calculate_mrr(keyword: str, retrieved_chunks: list[str]) -> float:
    """Reciprocal rank for a single keyword (case-insensitive)."""
    kw = keyword.lower()
    for rank, chunk in enumerate(retrieved_chunks, start=1):
        if kw in chunk.lower():
            return 1.0 / rank
    return 0.0


def calculate_dcg(relevances: list[int], k: int) -> float:
    """Discounted Cumulative Gain."""
    return sum(
        relevances[i] / math.log2(i + 2)
        for i in range(min(k, len(relevances)))
    )


def calculate_ndcg(keyword: str, retrieved_chunks: list[str], k: int = 10) -> float:
    """nDCG for a single keyword (binary relevance, case-insensitive)."""
    kw = keyword.lower()
    relevances = [1 if kw in chunk.lower() else 0 for chunk in retrieved_chunks[:k]]
    dcg  = calculate_dcg(relevances, k)
    idcg = calculate_dcg(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0.0


# Retrieval evaluator
def evaluate_retrieval(test: TestQuestion, k: int = 10) -> RetrievalEval:
    """Compute MRR, nDCG, and keyword coverage for a single test."""
    retrieved_chunks = fetch_context(test.question, k=k)

    mrr_scores  = [calculate_mrr(kw, retrieved_chunks)          for kw in test.keywords]
    ndcg_scores = [calculate_ndcg(kw, retrieved_chunks, k=k)    for kw in test.keywords]

    keywords_found   = sum(1 for s in mrr_scores if s > 0)
    total_keywords   = len(test.keywords)
    keyword_coverage = (keywords_found / total_keywords * 100) if total_keywords > 0 else 0.0

    return RetrievalEval(
        mrr              = sum(mrr_scores)  / len(mrr_scores)  if mrr_scores  else 0.0,
        ndcg             = sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0.0,
        keywords_found   = keywords_found,
        total_keywords   = total_keywords,
        keyword_coverage = keyword_coverage,
    )


# Answer evaluator (LLM-as-a-judge)
def evaluate_answer(test: TestQuestion) -> tuple[AnswerEval, str, list[str]]:
    """Generate an answer via RAG then score it with an LLM judge."""
    generated_answer, retrieved_chunks = answer_question(test.question)

    judge_messages = [
        {
            "role": "system",
            "content": (
                "You are an expert evaluator assessing the quality of answers. "
                "Evaluate the generated answer by comparing it to the reference answer. "
                "Only give 5/5 scores for perfect answers."
            ),
        },
        {
            "role": "user",
            "content": f"""Question:\n{test.question}\n\nGenerated Answer:\n{generated_answer}\n\nReference Answer:\n{test.reference_answer}\n\nPlease evaluate the generated answer on three dimensions:\n1. Accuracy: How factually correct is it compared to the reference answer? Only give 5/5 scores for perfect answers.\n2. Completeness: How thoroughly does it address all aspects of the question, covering all the information from the reference answer?\n3. Relevance: How well does it directly answer the specific question asked, giving no additional information?\n\nProvide detailed feedback and scores from 1 (very poor) to 5 (ideal) for each dimension. If the answer is wrong, then the accuracy score must be 1.""",
        },
    ]

    judge_response = completion(
        model=EVAL_MODEL,
        messages=judge_messages,
        response_format=AnswerEval,
    )
    answer_eval = AnswerEval.model_validate_json(judge_response.choices[0].message.content)
    return answer_eval, generated_answer, retrieved_chunks


print('Evaluation functions defined')

## Evaluation Dashboard (Gradio)

In [ ]:
# Thresholds 
MRR_GREEN      = 0.9;  MRR_AMBER      = 0.75
NDCG_GREEN     = 0.9;  NDCG_AMBER     = 0.75
COVERAGE_GREEN = 90.0; COVERAGE_AMBER = 75.0
ANSWER_GREEN   = 4.5;  ANSWER_AMBER   = 4.0


# Color helper 
def get_color(value: float, metric_type: str) -> str:
    thresholds = {
        "mrr":          (MRR_GREEN,      MRR_AMBER),
        "ndcg":         (NDCG_GREEN,     NDCG_AMBER),
        "coverage":     (COVERAGE_GREEN, COVERAGE_AMBER),
        "accuracy":     (ANSWER_GREEN,   ANSWER_AMBER),
        "completeness": (ANSWER_GREEN,   ANSWER_AMBER),
        "relevance":    (ANSWER_GREEN,   ANSWER_AMBER),
    }
    green, amber = thresholds.get(metric_type, (1.0, 0.5))
    return "#28a745" if value >= green else ("#fd7e14" if value >= amber else "#dc3545")


# Metric card HTML 
def metric_card(label: str, value: float, metric_type: str,
                is_percentage: bool = False, score_format: bool = False) -> str:
    color = get_color(value, metric_type)
    if is_percentage:  val_str = f"{value:.1f}%"
    elif score_format: val_str = f"{value:.2f} / 5"
    else:              val_str = f"{value:.4f}"
    return f"""
    <div style="margin:10px 0;padding:15px;background:#f8f9fa;
                border-radius:8px;border-left:5px solid {color};">
      <div style="font-size:13px;color:#6c757d;margin-bottom:4px;">{label}</div>
      <div style="font-size:28px;font-weight:700;color:{color};">{val_str}</div>
    </div>"""


def done_banner(count: int) -> str:
    return f"""
    <div style="margin-top:16px;padding:10px;background:#d4edda;border:1px solid #c3e6cb;
                border-radius:6px;text-align:center;">
      <span style="font-size:14px;color:#155724;font-weight:600;">
        ✓ Evaluation complete — {count} tests
      </span>
    </div>"""


# Generator wrappers (replicate eval.py evaluate_all_* style) 
def evaluate_all_retrieval():
    """Yield (test, RetrievalEval, progress_fraction) for every test."""
    tests = load_tests()
    total = len(tests)
    for i, test in enumerate(tests):
        result = evaluate_retrieval(test)
        yield test, result, (i + 1) / total


def evaluate_all_answers():
    """Yield (test, AnswerEval, progress_fraction) for every test."""
    tests = load_tests()
    total = len(tests)
    for i, test in enumerate(tests):
        ans_eval, _, _ = evaluate_answer(test)
        yield test, ans_eval, (i + 1) / total


# Gradio event handlers 
def run_retrieval_evaluation(progress=gr.Progress()):
    total_mrr = total_ndcg = total_coverage = 0.0
    category_mrr: dict[str, list[float]] = defaultdict(list)
    count = 0

    for test, result, prog in evaluate_all_retrieval():
        count += 1
        total_mrr      += result.mrr
        total_ndcg     += result.ndcg
        total_coverage += result.keyword_coverage
        category_mrr[test.category].append(result.mrr)
        progress(prog, desc=f"Retrieval — test {count}…")

    avg_mrr      = total_mrr      / count
    avg_ndcg     = total_ndcg     / count
    avg_coverage = total_coverage / count

    html = "<div style='padding:0;'>" + \
        metric_card("Mean Reciprocal Rank (MRR)",  avg_mrr,      "mrr") + \
        metric_card("Normalized DCG (nDCG)",        avg_ndcg,     "ndcg") + \
        metric_card("Keyword Coverage",             avg_coverage, "coverage", is_percentage=True) + \
        done_banner(count) + "</div>"

    df = pd.DataFrame([
        {"Category": cat, "Average MRR": sum(scores) / len(scores)}
        for cat, scores in category_mrr.items()
    ])
    return html, df


def run_answer_evaluation(progress=gr.Progress()):
    total_acc = total_comp = total_rel = 0.0
    category_acc: dict[str, list[float]] = defaultdict(list)
    count = 0

    for test, result, prog in evaluate_all_answers():
        count += 1
        total_acc  += result.accuracy
        total_comp += result.completeness
        total_rel  += result.relevance
        category_acc[test.category].append(result.accuracy)
        progress(prog, desc=f"Answer — test {count}…")

    avg_acc  = total_acc  / count
    avg_comp = total_comp / count
    avg_rel  = total_rel  / count

    html = "<div style='padding:0;'>" + \
        metric_card("Accuracy",     avg_acc,  "accuracy",     score_format=True) + \
        metric_card("Completeness", avg_comp, "completeness", score_format=True) + \
        metric_card("Relevance",    avg_rel,  "relevance",    score_format=True) + \
        done_banner(count) + "</div>"

    df = pd.DataFrame([
        {"Category": cat, "Average Accuracy": sum(scores) / len(scores)}
        for cat, scores in category_acc.items()
    ])
    return html, df


# Build and launch the dashboard 
IDLE_HTML = "<div style='padding:20px;text-align:center;color:#999;font-size:14px;'>" \
            "Click <b>Run Evaluation</b> to start</div>"

theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="myHR RAG Evaluation Dashboard", theme=theme) as eval_app:

    gr.Markdown("# 📊 myHR RAG Evaluation Dashboard")
    gr.Markdown("Evaluate retrieval and answer quality for the **Archery Canada HR Manual** RAG system.")

    # ── Retrieval section ──
    gr.Markdown("## 🔍 Retrieval Evaluation")
    gr.Markdown("Measures how well the FAISS retriever surfaces relevant chunks using MRR, nDCG, and keyword coverage.")
    retrieval_btn = gr.Button("Run Retrieval Evaluation", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=1):
            retrieval_metrics = gr.HTML(IDLE_HTML)
        with gr.Column(scale=1):
            retrieval_chart = gr.BarPlot(
                x="Category",
                y="Average MRR",
                title="Average MRR by Category",
                y_lim=[0, 1],
                height=350,
            )

    gr.Markdown("---")

    # Answer section 
    gr.Markdown("## 💬 Answer Evaluation")
    gr.Markdown("LLM-as-a-judge scores each generated answer on Accuracy, Completeness, and Relevance (1–5 scale).")
    answer_btn = gr.Button("Run Answer Evaluation", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=1):
            answer_metrics = gr.HTML(IDLE_HTML)
        with gr.Column(scale=1):
            answer_chart = gr.BarPlot(
                x="Category",
                y="Average Accuracy",
                title="Average Accuracy by Category",
                y_lim=[1, 5],
                height=350,
            )

    # Wire buttons 
    retrieval_btn.click(
        fn=run_retrieval_evaluation,
        outputs=[retrieval_metrics, retrieval_chart],
    )
    answer_btn.click(
        fn=run_answer_evaluation,
        outputs=[answer_metrics, answer_chart],
    )

eval_app.launch(share=False)